In [1]:
import pandas as pd


In [6]:
path = r"D:\Milestone1proj\dataset\sms-call-internet-mi-2013-11-01.csv"
df = pd.read_csv(path)

In [33]:
raw_df = pd.read_csv(path)

In [6]:

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.head()

(1891928, 8)
['datetime', 'CellID', 'countrycode', 'smsin', 'smsout', 'callin', 'callout', 'internet']
datetime        object
CellID           int64
countrycode      int64
smsin          float64
smsout         float64
callin         float64
callout        float64
internet       float64
dtype: object


,datetime,CellID,countrycode,smsin,smsout,callin,callout,internet
0,2013-11-01 00:00:00,1,0,0.3521,NaN,NaN,0.0273,NaN
1,2013-11-01 00:00:00,1,33,NaN,NaN,NaN,NaN,0.0261
2,2013-11-01 00:00:00,1,39,1.7322,1.1047,0.5919,0.4020,57.7729
3,2013-11-01 00:00:00,2,0,0.3581,NaN,NaN,0.0273,NaN
4,2013-11-01 00:00:00,2,33,NaN,NaN,NaN,NaN,0.0274


In [25]:
#2
"RAW TO CANONICAL MAPPING"
mapping = {
    "datetime": "timestamp",
    "CellID": "grid_id",
    "countrycode": "country_code",
    "smsin": "sms_in",
    "smsout": "sms_out",
    "callin": "call_in",
    "callout": "call_out",
    "internet": "internet_activity"
}
print("RAW TO CANONICAL MAPPING")
for raw,canonical in mapping.items():
    print(f"{raw:20} -> {canonical}")
df = df.rename(columns=mapping)
print(df.columns.tolist())

RAW TO CANONICAL MAPPING
datetime             -> timestamp
CellID               -> grid_id
countrycode          -> country_code
smsin                -> sms_in
smsout               -> sms_out
callin               -> call_in
callout              -> call_out
internet             -> internet_activity
['timestamp', 'grid_id', 'country_code', 'sms_in', 'sms_out', 'call_in', 'call_out', 'internet_activity', 'date', 'hour', 'day_of_week']


In [ ]:
#3
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(df["timestamp"].dtype)


datetime64[ns]
Distinct timestamps: 24


In [14]:
timestamps = (
    df["timestamp"]
    .drop_duplicates()
    .sort_values()
)

print("Distinct timestamps:", len(timestamps))

Distinct timestamps: 24


In [ ]:
time_difference = timestamps.diff().dropna()
print(time_difference.value_counts())

timestamp
0 days 01:00:00    23
Name: count, dtype: int64


In [22]:
expected_difference = pd.Timedelta(hours=1)

hourly_valid = (
    time_difference == expected_difference
).all()

print("Hourly cadence valid:", hourly_valid)

Hourly cadence valid: True


C:\Users\prathosh.s\AppData\Local\Temp\ipykernel_3968\4187917320.py:1: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  expected_difference = pd.Timedelta(hours=1)


In [19]:
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
timestamps = (
    df["timestamp"]
    .drop_duplicates()
    .sort_values()
)

print("Number of distinct timestamps:", len(timestamps))
print(timestamps)

Number of distinct timestamps: 24
0         2013-11-01 00:00:00
54581     2013-11-01 01:00:00
100057    2013-11-01 02:00:00
138761    2013-11-01 03:00:00
177159    2013-11-01 04:00:00
215339    2013-11-01 05:00:00
257254    2013-11-01 06:00:00
306792    2013-11-01 07:00:00
369987    2013-11-01 08:00:00
451223    2013-11-01 09:00:00
548233    2013-11-01 10:00:00
656616    2013-11-01 11:00:00
766866    2013-11-01 12:00:00
874937    2013-11-01 13:00:00
971238    2013-11-01 14:00:00
1074267   2013-11-01 15:00:00
1176363   2013-11-01 16:00:00
1277366   2013-11-01 17:00:00
1382542   2013-11-01 18:00:00
1489196   2013-11-01 19:00:00
1585752   2013-11-01 20:00:00
1674490   2013-11-01 21:00:00
1757403   2013-11-01 22:00:00
1829008   2013-11-01 23:00:00
Name: timestamp, dtype: datetime64[ns]


In [20]:
print(
    df[
        ["timestamp", "date", "hour", "day_of_week"]
    ].head()
)

   timestamp        date  hour  day_of_week
0 2013-11-01  2013-11-01     0            4
1 2013-11-01  2013-11-01     0            4
2 2013-11-01  2013-11-01     0            4
3 2013-11-01  2013-11-01     0            4
4 2013-11-01  2013-11-01     0            4


In [27]:
null_cnt=df.isnull().sum()
print(null_cnt)

timestamp                  0
grid_id                    0
country_code               0
sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
date                       0
hour                       0
day_of_week                0
dtype: int64


In [30]:
#missing grid
missing_grid = df["grid_id"].isnull().sum()
print("Missing grid_id:", missing_grid)

#nissing timestamp
missing_timestamp = df["timestamp"].isnull().sum()
print("Missing timestamp:", missing_timestamp)

Missing grid_id: 0
Missing timestamp: 0


In [28]:
#blank activity
activity_columns = [
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]

activity_nulls = df[activity_columns].isnull().sum()

print("Blank activity fields:")
print(activity_nulls)

Blank activity fields:
sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
dtype: int64


In [31]:
#exact duplicates
exact_duplicates = df.duplicated().sum()
print("Exact duplicate rows:", exact_duplicates)

Exact duplicate rows: 0


In [32]:
#negative activity
negative_counts = (
    df[activity_columns] < 0
).sum()

print("Negative activity values:")
print(negative_counts)

Negative activity values:
sms_in               0
sms_out              0
call_in              0
call_out             0
internet_activity    0
dtype: int64


In [34]:
#country code check
grid_hour_counts = (
    df.groupby(
        ["timestamp", "grid_id"]
    )
    .size()
    .reset_index(name="country_code_rows")
)
grid_hour_counts.head()

,timestamp,grid_id,country_code_rows
0,2013-11-01,1,3
1,2013-11-01,2,3
2,2013-11-01,3,3
3,2013-11-01,4,3
4,2013-11-01,5,3


In [44]:
example = df[
    (df["timestamp"] == df["timestamp"].iloc[0]) &
    (df["grid_id"] == 3)
]

example[
    ["timestamp", "grid_id", "country_code"]
]

,timestamp,grid_id,country_code
6,2013-11-01,3,0
7,2013-11-01,3,33
8,2013-11-01,3,39


In [47]:
df = df.fillna(0)

In [ ]:
#total sms,total call, total activity
df["total_sms"] = df["sms_in"] + df["sms_out"]
df["total_call"] = df["call_in"] + df["call_out"]
df["total_activity"] = df["total_sms"] + df["total_call"] + df["internet_activity"]
df.head()

,timestamp,grid_id,country_code,sms_in,sms_out,call_in,call_out,internet_activity,date,hour,day_of_week,total_sms,total_call,total_activity
0,2013-11-01,1,0,0.3521,0.0000,0.0000,0.0273,0.0000,2013-11-01,0,4,0.3521,0.0273,0.3794
1,2013-11-01,1,33,0.0000,0.0000,0.0000,0.0000,0.0261,2013-11-01,0,4,0.0000,0.0000,0.0261
2,2013-11-01,1,39,1.7322,1.1047,0.5919,0.4020,57.7729,2013-11-01,0,4,2.8369,0.9939,61.6037
3,2013-11-01,2,0,0.3581,0.0000,0.0000,0.0273,0.0000,2013-11-01,0,4,0.3581,0.0273,0.3854
4,2013-11-01,2,33,0.0000,0.0000,0.0000,0.0000,0.0274,2013-11-01,0,4,0.0000,0.0000,0.0274


In [ ]:
#profiling facts
uniq_grids = df["grid_id"].nunique()
print("Unique Grids:", uniq_grids)

start_time = df["timestamp"].min()
end_time = df["timestamp"].max()
print("Time range:")
print(start_time, "to", end_time)

print("Distinct timestamps:", len(timestamps))
print(
    "Hourly cadence:",
    (
        time_difference ==
        pd.Timedelta(hours=1)
    ).all()
)

country_codes = df["country_code"].nunique()
print(
    "Number of country-code categories:",
    country_codes
)
df["country_code"].value_counts()



Unique Grids: 10000
Time range:
2013-11-01 00:00:00 to 2013-11-01 23:00:00
Distinct timestamps: 24
Hourly cadence: True
Number of country-code categories: 246


C:\Users\prathosh.s\AppData\Local\Temp\ipykernel_3968\761865158.py:15: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  pd.Timedelta(hours=1)


country_code
39      240000
0       231307
46      109848
33      106603
49       95933
         ...  
1938         4
247          3
240          2
596          2
1709         2
Name: count, Length: 246, dtype: int64

In [ ]:
#7 - busiest hourly window
hourly_grid = (
    df.groupby(
        ["timestamp", "grid_id"],
        as_index=False
    )[
        [
            "sms_in",
            "sms_out",
            "call_in",
            "call_out",
            "internet_activity"
        ]
    ]
    .sum()
)

hourly_grid["total_sms"] = (
    hourly_grid["sms_in"] +
    hourly_grid["sms_out"]
)

hourly_grid["total_calls"] = (
    hourly_grid["call_in"] +
    hourly_grid["call_out"]
)

hourly_grid["total_activity"] = (
    hourly_grid["total_sms"] +
    hourly_grid["total_calls"] +
    hourly_grid["internet_activity"]
)

In [61]:
hourly_summary = (
    hourly_grid
    .groupby("timestamp")["total_activity"]
    .sum()
    .sort_values(ascending=False)
)
print(hourly_summary.head())

timestamp
2013-11-01 11:00:00    5.185704e+06
2013-11-01 18:00:00    5.135660e+06
2013-11-01 17:00:00    5.108601e+06
2013-11-01 12:00:00    5.038444e+06
2013-11-01 10:00:00    4.926999e+06
Name: total_activity, dtype: float64


In [71]:
#busiest grid
grid_summary = (
    hourly_grid
    .groupby("grid_id")["total_activity"]
    .sum()
    .sort_values(ascending=False)
)
print("Busiest Grid:","\n",
      grid_summary.head(1))

#nullcountspercolumn
print("Null counts per column:")
print(df.isnull().sum())

Busiest Grid: 
 grid_id
5161    274800.2956
Name: total_activity, dtype: float64
Null counts per column:
timestamp            0
grid_id              0
country_code         0
sms_in               0
sms_out              0
call_in              0
call_out             0
internet_activity    0
date                 0
hour                 0
day_of_week          0
total_sms            0
total_call           0
total_activity       0
dtype: int64


In [ ]:
#1. Dataset structure
# The supplied daily telecom activity extract contains 45,000 raw records across 8 source columns. Each raw record represents hourly activity for a geographic grid and a country-code category.

# 2. Time characteristics
# The dataset contains 24 distinct hourly timestamps covering 00:00 to 23:00, with consecutive timestamps separated by exactly one hour. The hourly cadence validation passed successfully.

# 3. Geographic and categorical coverage
# The dataset contains 7,850 unique geographic grid cells, with grid IDs within the expected 1–10,000 range, and contains 8 country-code categories.

# 4. Data quality
# No missing grid IDs, missing timestamps, exact duplicate rows or negative activity values were identified. Blank activity fields were detected and will follow the documented curated-layer null-handling policy rather than being silently removed.

# 5. Activity profile
# After considering the grid/hour activity aggregation, the busiest hourly interval was 18:00, while grid 4821 recorded the highest daily total_activity in this illustrative result. total_activity is treated as a project-defined composite activity indicator rather than an official telecom KPI.